# Session 6 — Hypothesis Testing

**Goal:** replace "these two numbers look different" with a procedure that says how
surprising the difference would be if there were no real effect — and, just as
importantly, learn what that procedure does *not* tell you.

## What this stage does for the system

Session 5 produced a ranked list of candidate predictors from 297 patients. Every
number on that list is an estimate, and Session 4 showed exactly how much estimates
wobble from sample to sample. So the open question is: **which of those relationships
would still be there in the next 297 patients, and which are this sample's noise?**

Hypothesis testing is the general framework for that question. Sessions 7 and 8 are the
two specific instruments (continuous comparisons, categorical comparisons); this
session is the framework they share, plus the four ways it gets misused — treating a
p-value as a probability the hypothesis is true, treating "not significant" as proof of
no effect, treating "significant" as "important", and running enough tests that
something is bound to come up significant.

The last of those is the one that scales into a systems problem. This registry has 13
candidate inputs, and screening all of them at $\alpha = 0.05$ makes a false positive
close to a coin flip — Step 7 computes it.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — State the hypotheses before looking at the answer

The claim to test: **do male and female patients in this registry have different
disease rates?**

- $H_0$ (null): the two rates are equal. Any observed gap is sampling noise.
- $H_1$ (alternative): the rates differ. Two-sided, because the direction was not
  predicted in advance.
- $\alpha = 0.05$: the false-positive rate we are willing to accept, **chosen now**,
  before seeing the result.

The ordering is not ceremony. A threshold picked after seeing the p-value is not a
threshold, and a direction chosen after seeing which way the data went doubles the
false-positive rate the test reports.

In [ ]:
ALPHA = 0.05

male_rate = df.loc[df["sex"] == 1, "target"].mean()
female_rate = df.loc[df["sex"] == 0, "target"].mean()

print(f"male patients:   {(df['sex'] == 1).sum():3}   disease rate {male_rate:.3f}")
print(f"female patients: {(df['sex'] == 0).sum():3}   disease rate {female_rate:.3f}")
print(f"observed gap:                     {male_rate - female_rate:+.3f}")
print()
print(f"H0: rates are equal | H1: rates differ | alpha = {ALPHA}")

**Observe:** 201 male patients at a `0.557` disease rate versus 96 female patients at
`0.260` — a gap of `+0.297`, nearly 30 percentage points.
**Infer:** a gap this large in a sample this size is unlikely to be noise, but "unlikely"
is exactly the word the test replaces with a number, so hold the conclusion. Note the
unequal group sizes: 201 versus 96 means the female rate is estimated from less than
half as much data and therefore carries a wider error bar — a fact the test accounts
for automatically, and one that becomes a design problem in Step 6. Note too what $H_0$
is: not "sex is irrelevant to heart disease" but the narrow, testable claim "these two
rates are equal in the population this sample came from".

## Step 3 — Compute the test statistic and its p-value

A **two-proportion z-test** compares two rates. The statistic is the observed gap
divided by its standard error under $H_0$ — i.e. "how many standard errors from zero
is this gap?" The p-value is the probability of a gap at least this extreme *if $H_0$
were true*.

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

counts = np.array([df.loc[df["sex"] == 1, "target"].sum(),
                   df.loc[df["sex"] == 0, "target"].sum()])
nobs = np.array([(df["sex"] == 1).sum(), (df["sex"] == 0).sum()])

z_stat, p_value = proportions_ztest(counts, nobs)
print(f"disease counts: {counts},  group sizes: {nobs}")
print(f"z = {z_stat:.3f}")
print(f"p = {p_value:.2e}")
print()
print(f"p < alpha ({ALPHA})? {p_value < ALPHA}  ->  {'reject H0' if p_value < ALPHA else 'fail to reject H0'}")

**Observe:** `z = 4.799` and `p = 1.59e-06` — the observed gap sits nearly five standard
errors from zero, and $H_0$ is rejected.
**Infer:** read that p-value precisely: *if the two rates were truly equal, a gap this
large or larger would occur about twice in a million samples.* It is emphatically **not**
"a two-in-a-million chance that the rates are equal" — the p-value is a probability
about the data given the hypothesis, never about the hypothesis given the data, and
that inversion is the single most common misreading in applied statistics (it is the
same inversion Session 1's Bayes calculation warned about, in a different costume).
Note also what rejection does not deliver: it says the gap is not zero, and says nothing
about whether it is large enough to matter. Step 5 is where that gets addressed.

## Step 4 — The same test on an input that carries no signal

A framework that only ever rejects is not a framework. Session 1 found `fbs` to be
practically independent of disease; running the identical procedure on it shows what
"fail to reject" looks like.

In [ ]:
fbs_counts = np.array([df.loc[df["fbs"] == 1, "target"].sum(),
                       df.loc[df["fbs"] == 0, "target"].sum()])
fbs_nobs = np.array([(df["fbs"] == 1).sum(), (df["fbs"] == 0).sum()])

z_fbs, p_fbs = proportions_ztest(fbs_counts, fbs_nobs)
print(f"fbs=1 rate: {fbs_counts[0] / fbs_nobs[0]:.3f}  (n={fbs_nobs[0]})")
print(f"fbs=0 rate: {fbs_counts[1] / fbs_nobs[1]:.3f}  (n={fbs_nobs[1]})")
print(f"z = {z_fbs:.3f}, p = {p_fbs:.3f}  ->  {'reject H0' if p_fbs < ALPHA else 'fail to reject H0'}")

**Observe:** rates of `0.465` and `0.461`, `z = 0.055`, `p = 0.956` — as close to "no
evidence whatever" as a test result comes.
**Infer:** the correct phrasing is **"fail to reject $H_0$"**, not "accept $H_0$", and
the distinction is not pedantry. This result is equally consistent with two very
different worlds: `fbs` genuinely has no effect, or it has a modest effect that 43
`fbs = 1` patients cannot detect. Nothing here distinguishes them — that is what Step 6
quantifies. The practical consequence for the system: a non-significant result is weak
grounds for permanently discarding an input, especially one that is cheap to measure
and clinically plausible. Park it and revisit when the registry grows, rather than
concluding it is useless.

## Step 5 — Effect size: significant is not the same as important

The p-value confounds two things — how big the effect is and how much data you have.
An **effect size** separates them out by measuring the gap in standard units,
independent of sample size. For proportions, Cohen's $h$: roughly 0.2 is small, 0.5
medium, 0.8 large.

In [ ]:
from statsmodels.stats.proportion import proportion_effectsize

h_sex = proportion_effectsize(male_rate, female_rate)
h_fbs = proportion_effectsize(fbs_counts[0] / fbs_nobs[0], fbs_counts[1] / fbs_nobs[1])

print(f"sex: gap {male_rate - female_rate:+.3f}, Cohen's h = {h_sex:+.3f}, p = {p_value:.2e}")
print(f"fbs: gap {fbs_counts[0]/fbs_nobs[0] - fbs_counts[1]/fbs_nobs[1]:+.3f}, "
      f"Cohen's h = {h_fbs:+.3f}, p = {p_fbs:.3f}")
print()

# The same tiny effect becomes "significant" with enough patients.
print("A gap of exactly 1 percentage point (0.46 vs 0.47), tested at growing n:")
tiny = proportion_effectsize(0.47, 0.46)
for n in [300, 3_000, 30_000, 300_000]:
    counts_n = np.array([int(0.47 * n), int(0.46 * n)])
    _, p_n = proportions_ztest(counts_n, np.array([n, n]))
    print(f"  n={n:>7} per group: h = {tiny:.4f} (unchanged), p = {p_n:.4f}"
          f"{'  <- significant' if p_n < ALPHA else ''}")

**Observe:** `sex` has `h = 0.614` (medium-to-large) with a tiny p-value, `fbs` has
`h = 0.009` with a p-value near 1 — and in the second block, a fixed one-point gap
keeps an effect size of `0.0200` at every sample size while its p-value marches down
past 0.05 somewhere around n = 30,000.
**Infer:** the second block is the argument for never reporting a p-value alone. With
enough patients, *any* non-zero difference becomes statistically significant, so on a
large enough registry "p < 0.05" degenerates into "the effect is not exactly zero" —
which is nearly always true and nearly never useful. Effect size is what stays
constant, and it is the quantity a clinician actually needs: a one-point difference in
disease rate does not change a referral decision no matter how many decimal places of
significance it carries. Report both, always, and let the effect size carry the
decision.

## Step 6 — Type I and Type II errors, and statistical power

Two ways to be wrong: **Type I** (reject a true $H_0$ — a false alarm, controlled at
$\alpha$) and **Type II** (fail to reject a false $H_0$ — a missed effect, with rate
$\beta$). **Power** is $1 - \beta$: the probability of detecting an effect that is
genuinely there. The convention is to aim for 0.80.

In [ ]:
from statsmodels.stats.power import NormalIndPower

power_analysis = NormalIndPower()

power_sex = power_analysis.power(effect_size=abs(h_sex), nobs1=nobs[0],
                                 ratio=nobs[1] / nobs[0], alpha=ALPHA)
print(f"power to detect the sex effect (h={abs(h_sex):.3f}) at these group sizes: {power_sex:.3f}")

print("\nPower to detect a SMALL effect (h = 0.2), by group size:")
for n in [43, 100, 300, 400, 800]:
    pw = power_analysis.power(effect_size=0.2, nobs1=n, ratio=1.0, alpha=ALPHA)
    print(f"  n={n:>4} per group: power = {pw:.3f}{'  <- adequate' if pw >= 0.8 else ''}")

n_needed = power_analysis.solve_power(effect_size=0.2, power=0.8, alpha=ALPHA)
print(f"\npatients per group needed for 80% power on a small effect: {n_needed:.0f}")
print(f"this registry's smallest group (fbs=1): {fbs_nobs[0]}")

**Observe:** power to detect the sex effect is `0.999` — essentially guaranteed — while
detecting a *small* effect (h = 0.2) needs `392` patients per group, and the
`fbs = 1` group has 43.
**Infer:** this retroactively explains Step 4. With 43 patients in one group, power
against a small effect is `0.153`, meaning that even if `fbs` did have a small real
effect, this test would miss it five times out of six. So `p = 0.956` is not
evidence of absence — it is a test that was never capable of finding what it was
looking for. This is the single most useful thing power analysis does, and the reason
to run it **before** collecting data rather than after: a study underpowered for the
effect it cares about produces uninterpretable non-results, and no analysis afterwards
can repair that. Note also the asymmetry $\alpha$ vs $\beta$ encodes — the convention
of 0.05 and 0.20 says a false alarm is four times more costly than a missed effect,
which is a value judgement, and for a screening tool where a missed case is the worse
outcome it may well be the wrong one.

## Step 7 — The multiple-comparisons trap

Every test at $\alpha = 0.05$ carries a 5% false-positive rate. Run many, and the
probability that *at least one* comes back significant by chance alone is
$1 - (1-\alpha)^k$ — which grows fast.

In [ ]:
candidates = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
              "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

print(f"{'tests':>6} {'P(at least one false positive)':>32}")
for k in [1, 5, 13, 50]:
    print(f"{k:>6} {1 - 0.95 ** k:>32.3f}")
print(f"\nthis registry has {len(candidates)} candidate inputs")

**Observe:** one test gives the promised `0.050`; thirteen gives `0.487`.
**Infer:** screening this registry's inputs at $\alpha = 0.05$ is close to a coin flip
on whether at least one "significant" finding is pure noise — and the finding will not
be labelled, so you cannot tell which. This is the mechanism behind p-hacking, and it
does not require any bad faith: trying a few subgroups, a few transformations, a few
thresholds, and reporting what worked is the same arithmetic. The rule that follows is
about *bookkeeping*: count every test you ran, not just the ones you reported.

In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy import stats

# Screen every candidate input against the outcome with an appropriate test.
rows = []
for col in candidates:
    if df[col].nunique() <= 2:                      # binary -> two-proportion z-test
        c = np.array([df.loc[df[col] == df[col].max(), "target"].sum(),
                      df.loc[df[col] != df[col].max(), "target"].sum()])
        n = np.array([(df[col] == df[col].max()).sum(), (df[col] != df[col].max()).sum()])
        _, p = proportions_ztest(c, n)
        test = "z-test"
    elif df[col].nunique() <= 5:                    # few categories -> chi-square
        _, p, _, _ = stats.chi2_contingency(pd.crosstab(df[col], df["target"]))
        test = "chi-square"
    else:                                           # continuous -> Welch's t-test
        _, p = stats.ttest_ind(df.loc[df["target"] == 1, col],
                               df.loc[df["target"] == 0, col], equal_var=False)
        test = "t-test"
    rows.append({"input": col, "test": test, "p": p})

screen = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
screen["raw < 0.05"] = screen["p"] < ALPHA
screen["bonferroni"] = multipletests(screen["p"], alpha=ALPHA, method="bonferroni")[0]
screen["BH (FDR)"] = multipletests(screen["p"], alpha=ALPHA, method="fdr_bh")[0]

pd.set_option("display.float_format", lambda v: f"{v:.2e}")
print(screen.to_string(index=False))
pd.reset_option("display.float_format")
print(f"\nsurvive raw alpha: {screen['raw < 0.05'].sum()} | "
      f"Bonferroni: {screen['bonferroni'].sum()} | BH: {screen['BH (FDR)'].sum()}")

**Observe:** eleven of thirteen inputs clear the raw threshold; Bonferroni keeps nine,
dropping the two borderline ones (`restecg` at `8.3e-03` and `trestbps` at `8.8e-03`),
while BH keeps all eleven. `chol` and `fbs` fail under every column.
**Infer:** nine of eleven survive the strictest correction available, and that is itself
the finding: when effects are genuinely strong (Session 5 showed six inputs correlating
above 0.4 with the outcome), multiple-comparison corrections cost you almost nothing,
so there is no reason to skip them. The two methods control different things and the
choice follows from purpose. **Bonferroni** divides $\alpha$ by the number of tests to
bound the chance of *any* false positive — the right default for a confirmatory claim
where one wrong finding discredits the result. **Benjamini-Hochberg** bounds the
expected *proportion* of false positives among what you flag, which is the right
default for screening, where the output is a shortlist for further investigation and a
few false leads are an acceptable price for missing fewer real ones. This is a
screening step feeding Session 9, so BH is the appropriate choice.

## What this session hands to the next one

- **A tested shortlist**, with p-values corrected for the 13 comparisons that produced
  it — this is what Session 9 draws its regression inputs from.
- **The framework itself** — hypotheses first, $\alpha$ first, effect size alongside
  every p-value — which Sessions 7 and 8 instantiate for continuous and categorical
  comparisons.
- **A power caveat** attached to every non-result: this registry cannot detect small
  effects in small subgroups, so `fbs` is parked, not dismissed.
- **The reminder from Session 5 that none of this is causal.** A corrected p-value
  establishes that a relationship is unlikely to be sampling noise. It says nothing
  about direction of cause.

Session 7 narrows the framework to the most common concrete case: comparing a
continuous measurement between two groups.

## Try it yourself

1. Set `ALPHA = 0.01` and re-run Steps 3-7. Which inputs drop out, and what has happened
   to the power figures in Step 6?
2. In Step 5, find the sample size at which a gap of 0.5 percentage points becomes
   significant. Would you report that finding?
3. Run the Step 3 test on `exang` and compute its Cohen's h. How does it rank against
   `sex` on effect size versus on p-value — and do the two orderings agree?
4. Add ten pure-noise columns (`rng.normal(size=len(df))`) to the Step 7 screen. How
   many clear the raw threshold, and does Bonferroni catch them all?